In [0]:
df_clean = spark.read.table("rides_clean")

from pyspark.sql.functions import sum, count, avg

driver_gold = df_clean.groupBy("driver_id").agg(
    sum("fare").alias("total_earnings"),
    count("trip_id").alias("total_trips"),
    avg("fare").alias("avg_fare")
)

user_gold = df_clean.groupBy("user_id").agg(
    sum("fare").alias("total_spent"),
    count("trip_id").alias("total_trip")
)

from pyspark.sql.functions import hour

time_gold = df_clean.withColumn("hour", hour("event_time")) \
    .groupBy("hour") \
    .agg(sum("fare").alias("revenue"))

driver_gold.write.mode("overwrite").format("delta").saveAsTable("driver_gold")

user_gold.write.mode("overwrite").format("delta").saveAsTable("user_gold")

time_gold.write.mode("overwrite").format("delta").saveAsTable("time_gold")